# 💻 Ejercicios Computacionales: Transformaciones Lineales

En esta sección pondrás en práctica los conceptos del capítulo usando **Python, NumPy y Matplotlib**. Cada ejercicio tiene código inicial (*starter code*) que debes completar.

> 🎯 **Objetivo:** Implementar y explorar transformaciones lineales computacionalmente.

---
## 🟦 Ejercicio 1: Función de rotación general

Implementa la función `rotacion(theta)` que devuelva la matriz de rotación de ángulo $\theta$ (en grados). Luego:
- Verifica que $R_{\theta} \cdot R_{-\theta} = I$
- Verifica que $\det(R_{\theta}) = 1$ para varios valores de $\theta$
- Aplica la rotación a los vectores de la base estándar y visualiza el resultado

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def rotacion(theta_grados):
    """Devuelve la matriz de rotacion 2x2 para el angulo theta en GRADOS."""
    theta = np.radians(theta_grados)  # convertir a radianes
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])

# --- Prueba 1: R_theta @ R_{-theta} debe ser la identidad ---
for angulo in [30, 45, 60, 90, 120, 180]:
    R_pos = rotacion(angulo)
    R_neg = rotacion(-angulo)
    producto = R_pos @ R_neg
    es_identidad = np.allclose(producto, np.eye(2))
    print(f"R_{angulo} @ R_{{-{angulo}}} = I? {es_identidad}")

# --- Prueba 2: det(R_theta) = 1 siempre ---
print()
angulos = np.linspace(0, 360, 37)
dets = [np.linalg.det(rotacion(a)) for a in angulos]
print(f"det siempre = 1: {np.allclose(dets, 1.0)}")

# --- Visualizacion: base estandar rotada por distintos angulos ---
e1 = np.array([1.0, 0.0])
e2 = np.array([0.0, 1.0])

fig, ax = plt.subplots(figsize=(7, 7))
ax.set_aspect('equal')
ax.axhline(0, color='gray', lw=0.5)
ax.axvline(0, color='gray', lw=0.5)

angulos_vis = [0, 30, 60, 90, 120, 150]
cmap = plt.cm.plasma

for i, ang in enumerate(angulos_vis):
    R = rotacion(ang)
    re1 = R @ e1
    col = cmap(i / len(angulos_vis))
    ax.annotate('', xy=re1, xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=col, lw=2))
    ax.text(re1[0]*1.1, re1[1]*1.1, f'{ang}°', fontsize=9, color=col)

# Circulo unitario
t = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(t), np.sin(t), 'gray', lw=0.8, linestyle='--', alpha=0.5)
ax.set_xlim(-1.4, 1.4)
ax.set_ylim(-1.4, 1.4)
ax.set_title('Rotacion del vector $e_1 = (1,0)$ por distintos angulos', fontsize=12)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

---
## 🟩 Ejercicio 2: Espacio nulo (núcleo) con descomposición SVD

Dada la matriz:
$$A = \begin{pmatrix} 1 & 2 & -1 & 0 \\ 2 & 4 & -1 & 1 \\ -1 & -2 & 2 & -1 \end{pmatrix}$$

Encuentra una base para el núcleo usando SVD. Verifica que $A\mathbf{v} \approx \mathbf{0}$ para cada vector base del núcleo.

In [ ]:
import numpy as np

A = np.array([[ 1,  2, -1,  0],
              [ 2,  4, -1,  1],
              [-1, -2,  2, -1]], dtype=float)

m, n = A.shape
print(f"Matriz A ({m}x{n}):")
print(A)

# --- Calcular SVD ---
# np.linalg.svd devuelve U (m x m), s (min(m,n),), Vt (n x n)
U, s, Vt = np.linalg.svd(A, full_matrices=True)

print(f"\nValores singulares: {np.round(s, 6)}")

# El rango es el numero de valores singulares significativamente distintos de cero
tolerancia = 1e-10
rango = np.sum(s > tolerancia)
nulidad = n - rango
print(f"Rango = {rango}, Nulidad = {nulidad}")

# Las ultimas 'nulidad' filas de Vt forman el nucleo
nucleo_base = Vt[rango:, :].T  # columnas = vectores del nucleo
print(f"\nBase del nucleo ({n}x{nulidad}):")
print(np.round(nucleo_base, 6))

# Verificacion: A @ v debe ser ~ 0
print("\nVerificacion A @ base_nucleo (debe ser ~0):")
residuo = A @ nucleo_base
print(np.round(residuo, 10))
print(f"Norma maxima del residuo: {np.max(np.abs(residuo)):.2e}")

---
## 🟨 Ejercicio 3: Composición de transformaciones

Sean:
- $T_1$: rotación de $45°$
- $T_2$: escala por factor $2$ en $x$ y $0.5$ en $y$

Verifica computacionalmente que $(T_2 \circ T_1)(\mathbf{x}) = T_2(T_1(\mathbf{x})) = (A_2 A_1)\mathbf{x}$ para varios vectores $\mathbf{x}$. Luego visualiza el efecto de la composición.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Definir las matrices
theta = np.radians(45)
A1 = np.array([[np.cos(theta), -np.sin(theta)],  # Rotacion 45 grados
               [np.sin(theta),  np.cos(theta)]])

A2 = np.array([[2.0, 0.0],   # Escala: x*2, y*0.5
               [0.0, 0.5]])

# Composicion: T2 despues de T1
A_comp = A2 @ A1

print("A1 (Rotacion 45 grados):")
print(np.round(A1, 4))
print("\nA2 (Escala):")
print(A2)
print("\nA2 @ A1 (Composicion T2 o T1):")
print(np.round(A_comp, 4))

# Verificacion para varios vectores aleatorios
np.random.seed(7)
vectores_prueba = np.random.randn(2, 10)

print("\nVerificacion (A2 @ A1) @ x == A2 @ (A1 @ x):")
for i in range(vectores_prueba.shape[1]):
    x = vectores_prueba[:, i]
    metodo1 = A_comp @ x
    metodo2 = A2 @ (A1 @ x)
    ok = np.allclose(metodo1, metodo2)
    print(f"  Vector {i+1}: {ok}", end="  ")
print()

# Visualizacion sobre el cuadrado unitario
cuadrado = np.array([[0,1,1,0,0],[0,0,1,1,0]], dtype=float)
cuad_T1 = A1 @ cuadrado
cuad_T2T1 = A_comp @ cuadrado

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
datos = [cuadrado, cuad_T1, cuad_T2T1]
titulos = ['Original', '$T_1$ (Rotacion 45°)', '$T_2 \\circ T_1$ (Compuesta)']
colores = ['steelblue', 'orange', 'tomato']

for ax, d, t, col in zip(axes, datos, titulos, colores):
    ax.fill(d[0], d[1], alpha=0.4, color=col)
    ax.plot(d[0], d[1], color=col, lw=2)
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.axhline(0, color='gray', lw=0.5)
    ax.axvline(0, color='gray', lw=0.5)
    ax.set_title(t, fontsize=11)
    ax.grid(True, alpha=0.3)

plt.suptitle('Composición de transformaciones: rotar luego escalar', fontsize=12)
plt.tight_layout()
plt.show()

---
## 🟥 Ejercicio 4: Verificar numéricamente que $f(x,y)=(x+y,\, xy)$ no es lineal

La función $f: \mathbb{R}^2 \to \mathbb{R}^2$ definida por $f(x,y) = (x+y,\; xy)$ parece "razonable", pero no es lineal. Demuéstralo numéricamente encontrando un contraejemplo para la aditividad.

In [ ]:
import numpy as np

def f(v):
    """f(x,y) = (x+y, x*y) — NO es lineal."""
    x, y = v
    return np.array([x + y, x * y])

# --- Probar aditividad: f(u+v) ?= f(u)+f(v) ---
print("Prueba de ADITIVIDAD para f(x,y) = (x+y, xy)\n")
pares = [
    (np.array([1.0, 2.0]), np.array([3.0, 4.0])),
    (np.array([0.0, 1.0]), np.array([1.0, 0.0])),
    (np.array([2.0,-1.0]), np.array([-1.0, 3.0])),
]

for u, v in pares:
    lhs = f(u + v)
    rhs = f(u) + f(v)
    ok  = np.allclose(lhs, rhs)
    print(f"u={u}, v={v}")
    print(f"  f(u+v) = {lhs}")
    print(f"  f(u)+f(v) = {rhs}")
    print(f"  ¿Iguales? {ok}  <-- {'OK' if ok else 'CONTRAEJEMPLO encontrado!'}")
    print()

# --- Probar homogeneidad: f(c*u) ?= c*f(u) ---
print("Prueba de HOMOGENEIDAD\n")
u = np.array([2.0, 3.0])
for c in [2, 3, -1, 0.5]:
    lhs = f(c * u)
    rhs = c * f(u)
    ok  = np.allclose(lhs, rhs)
    print(f"c={c}: f(cu)={lhs}, c*f(u)={rhs}, ¿Iguales? {ok}")

print("\nConclusión: el termino 'xy' es no lineal (producto de componentes).")
print("Una funcion lineal solo puede tener sumas y productos por escalares.")

---
## 🟪 Ejercicio 5: Construir la matriz de una transformación en $\mathbb{R}^3$

Sea $T: \mathbb{R}^3 \to \mathbb{R}^3$ definida por su acción sobre la base estándar:
$$T(\mathbf{e}_1) = (1, 2, 0), \quad T(\mathbf{e}_2) = (-1, 1, 3), \quad T(\mathbf{e}_3) = (2, 0, -1)$$

1. Construye la matriz $A$ de $T$.
2. Calcula $T(1, 1, 1)$ y $T(2, -1, 3)$ usando $A$.
3. Determina si $T$ es inyectiva, suprayectiva o biyectiva.
4. Si $T$ es invertible, calcula $A^{-1}$ y verifica $A A^{-1} = I$.

In [ ]:
import numpy as np

# --- 1. Construir la matriz A ---
# Las columnas de A son T(e1), T(e2), T(e3)
Te1 = np.array([1.0, 2.0, 0.0])
Te2 = np.array([-1.0, 1.0, 3.0])
Te3 = np.array([2.0, 0.0, -1.0])

A = np.column_stack([Te1, Te2, Te3])
print("Matriz A de la transformacion T:")
print(A)

# --- 2. Calcular T para vectores especificos ---
x1 = np.array([1.0, 1.0, 1.0])
x2 = np.array([2.0, -1.0, 3.0])
print(f"\nT(1,1,1)    = A @ x1 = {A @ x1}")
print(f"T(2,-1,3)   = A @ x2 = {A @ x2}")

# --- 3. Clasificar la transformacion ---
rango   = np.linalg.matrix_rank(A)
n       = A.shape[1]
m       = A.shape[0]
nulidad = n - rango
det_A   = np.linalg.det(A)

print(f"\nRango={rango}, Nulidad={nulidad}, det(A)={det_A:.4f}")
print(f"Inyectiva:    {nulidad == 0}")
print(f"Suprayectiva: {rango == m}")
print(f"Biyectiva:    {nulidad == 0 and rango == m}")

# --- 4. Inversa (si existe) ---
if np.abs(det_A) > 1e-10:
    A_inv = np.linalg.inv(A)
    print("\nMatriz inversa A^{-1}:")
    print(np.round(A_inv, 4))
    print("\nVerificacion A @ A^{-1} = I:")
    print(np.round(A @ A_inv, 6))
    print(f"¿Es la identidad? {np.allclose(A @ A_inv, np.eye(3))}")
else:
    print("\nA es singular, T no es invertible.")